1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Project imports
from database.metadata_store import MetadataStore
from database.vector_store import VectorStoreManager
from utils.config import config

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline

2. Initialize Stores

In [ ]:
# Initialize metadata and vector stores
metadata_store = MetadataStore()
vector_store = VectorStoreManager()

print("✓ Stores initialized successfully")
print(f"Metadata DB: {config.metadata_db_path}")
print(f"Vector DB: {config.vectordb_dir}")

3. Database Statistics

In [ ]:
# Get overall statistics
stats = metadata_store.get_statistics()

print("=" * 50)
print("DATABASE STATISTICS")
print("=" * 50)

for key, value in stats.items():
    print(f"{key:.<40} {value}")

print("=" * 50)

4. Document Analysis

Load Documents

In [ ]:
documents = metadata_store.get_all_documents()

if documents:
    df_docs = pd.DataFrame(documents)

    print(f"Total documents: {len(df_docs)}")
    print("\nFirst 5 documents:")

    display(df_docs.head())

else:
    print("⚠️ No documents found in database")
    df_docs = pd.DataFrame()

Document Statistics

In [ ]:
if not df_docs.empty:
    print("\nDocument Statistics:")
    print(
        df_docs[
            ['num_pages', 'num_chunks', 'file_size']
        ].describe()
    )

5. Document Status Distribution

In [ ]:
if not df_docs.empty and 'status' in df_docs.columns:

    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(14, 5)
    )

    status_counts = df_docs['status'].value_counts()

    # Bar Chart
    status_counts.plot(
        kind='bar',
        ax=ax1,
        color='skyblue'
    )

    ax1.set_title(
        'Document Status Distribution',
        fontsize=14,
        fontweight='bold'
    )

    ax1.set_xlabel('Status')
    ax1.set_ylabel('Count')
    ax1.tick_params(axis='x', rotation=45)

    # Pie Chart
    status_counts.plot(
        kind='pie',
        ax=ax2,
        autopct='%1.1f%%',
        startangle=90
    )

    ax2.set_title(
        'Status Percentage',
        fontsize=14,
        fontweight='bold'
    )

    ax2.set_ylabel('')

    plt.tight_layout()
    plt.show()

    print("\nStatus Breakdown:")

    for status, count in status_counts.items():
        percentage = (count / len(df_docs)) * 100
        print(
            f"{status:.<20} "
            f"{count:>5} "
            f"({percentage:>5.1f}%)"
        )

6. File Type Analysis

In [ ]:
if not df_docs.empty and 'file_type' in df_docs.columns:

    type_counts = df_docs['file_type'].value_counts()

    plt.figure(figsize=(10, 6))

    type_counts.plot(
        kind='barh',
        color='coral'
    )

    plt.title(
        'Document Types Distribution',
        fontsize=14,
        fontweight='bold'
    )

    plt.xlabel('Count')
    plt.ylabel('File Type')

    plt.tight_layout()
    plt.show()

    print("\nFile Type Breakdown:")

    for ftype, count in type_counts.items():
        print(f"{ftype:.<20} {count:>5}")

7. Pages and Chunks Analysis

In [ ]:
if not df_docs.empty and 'num_pages' in df_docs.columns:

    fig, axes = plt.subplots(
        2, 2,
        figsize=(14, 10)
    )

    # Pages Distribution
    df_docs['num_pages'].hist(
        bins=20,
        ax=axes[0, 0],
        color='lightblue',
        edgecolor='black'
    )

    axes[0, 0].set_title(
        'Pages per Document'
    )

    # Chunks Distribution
    if 'num_chunks' in df_docs.columns:

        df_docs['num_chunks'].hist(
            bins=20,
            ax=axes[0, 1],
            color='lightgreen',
            edgecolor='black'
        )

        axes[0, 1].set_title(
            'Chunks per Document'
        )

    # Boxplots
    df_docs.boxplot(
        column='num_pages',
        ax=axes[1, 0]
    )

    if 'num_chunks' in df_docs.columns:
        df_docs.boxplot(
            column='num_chunks',
            ax=axes[1, 1]
        )

    plt.tight_layout()
    plt.show()

    print("\nPages Statistics:")
    print(df_docs['num_pages'].describe())

    if 'num_chunks' in df_docs.columns:
        print("\nChunks Statistics:")
        print(df_docs['num_chunks'].describe())

8. File Size Analysis

In [ ]:
if not df_docs.empty and 'file_size' in df_docs.columns:

    df_docs['file_size_mb'] = (
        df_docs['file_size'] /
        (1024 * 1024)
    )

    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(14, 5)
    )

    df_docs['file_size_mb'].hist(
        bins=20,
        ax=ax1,
        color='plum',
        edgecolor='black'
    )

    df_docs.boxplot(
        column='file_size_mb',
        ax=ax2
    )

    plt.tight_layout()
    plt.show()

    print("\nFile Size Statistics (MB):")
    print(df_docs['file_size_mb'].describe())

    print(
        f"\nTotal storage used: "
        f"{df_docs['file_size_mb'].sum():.2f} MB"
    )

9. Upload Timeline

In [ ]:
if not df_docs.empty and 'upload_date' in df_docs.columns:

    df_docs['upload_datetime'] = pd.to_datetime(
        df_docs['upload_date']
    )

    df_docs['upload_day'] = (
        df_docs['upload_datetime']
        .dt.date
    )

    uploads_per_day = (
        df_docs.groupby('upload_day')
        .size()
    )

    plt.figure(figsize=(14, 6))

    uploads_per_day.plot(
        kind='line',
        marker='o',
        linewidth=2
    )

    plt.title(
        'Document Uploads Over Time'
    )

    plt.show()

10. Query Analysis

Load Query Logs

In [ ]:
queries = metadata_store.get_queries(limit=1000)

if queries:

    df_queries = pd.DataFrame(queries)

    print(
        f"Total queries: "
        f"{len(df_queries)}"
    )

    display(
        df_queries[
            ['query_text',
             'response_time',
             'query_date']
        ].head()
    )

else:
    print("⚠️ No queries found")
    df_queries = pd.DataFrame()

Query Performance

In [ ]:
if not df_queries.empty:

    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(14, 5)
    )

    df_queries['response_time'].hist(
        bins=30,
        ax=ax1,
        color='orange'
    )

    if 'query_date' in df_queries.columns:

        df_queries['query_datetime'] = (
            pd.to_datetime(
                df_queries['query_date']
            )
        )

        (
            df_queries
            .set_index('query_datetime')
            ['response_time']
            .plot(ax=ax2)
        )

    plt.tight_layout()
    plt.show()

11. Vector Database Analysis

In [ ]:
vector_count = vector_store.count()

print("=" * 50)
print("VECTOR DATABASE STATISTICS")
print("=" * 50)

print(f"Total vectors: {vector_count}")

print(
    f"Embedding dimension: "
    f"{vector_store.embeddings.client._model_kwargs.get('dimensions', 1536)}"
)

print("=" * 50)

12. Top Documents by Chunks

In [ ]:
if not df_docs.empty and 'num_chunks' in df_docs.columns:

    top_docs = df_docs.nlargest(
        10,
        'num_chunks'
    )[
        ['filename',
         'num_pages',
         'num_chunks']
    ]

    display(top_docs)

13. Summary Report

In [ ]:
print("=" * 70)
print("SUMMARY REPORT".center(70))
print("=" * 70)

# Documents Summary
# Queries Summary
# Vector Database Summary

print(
    f"Report generated: "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)

14. Export Data

In [ ]:
from pathlib import Path

export_path = Path.cwd() / "exports"
export_path.mkdir(exist_ok=True)

if not df_docs.empty:

    docs_file = (
        export_path /
        f"documents_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )

    df_docs.to_csv(
        docs_file,
        index=False
    )

    print(
        f"✓ Documents exported to: "
        f"{docs_file}"
    )

if not df_queries.empty:

    queries_file = (
        export_path /
        f"queries_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )

    df_queries.to_csv(
        queries_file,
        index=False
    )

    print(
        f"✓ Queries exported to: "
        f"{queries_file}"
    )